# 06. Trajectory Collection and Error Slices

- Parent issue: `#22`
- Status: `active`
- Summary: Classify baseline EvalRecord failures into error types, produce retry-candidate set for synthetic data generation (#24), and write a category × error-type taxonomy report.

## Audience and Why It Matters

Model developers and reviewers deciding where the next engineering dollar should go.


## Decision / Hypothesis

Log raw completions, extracted answers, and correctness labels per category so the team can target the most recoverable failures first.


## Environment and Reproduction

- Python: 3.11+
- Run from repo root: `jupyter notebook notebooks/06_trajectory_collection_and_error_slices.ipynb`
- Requires: baseline `eval_records.jsonl` from issue `#19` run (set `BASELINE_PATH` below)
- Hardware: Colab A100 ~2 hr for full inference pass; RTX 3080 local ~4 hr
- Companion registry entry: `docs/execution/NOTEBOOKS.md`

In [ ]:
from pathlib import Path
import platform

REPO_ROOT = Path.cwd()

print(f"Repo root: {REPO_ROOT}")
print(f"Python platform: {platform.platform()}")


In [ ]:
# Device detection: prefers CUDA, falls back to CPU gracefully.
import sys
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir() and (_candidate / 'notebooks').is_dir():
        REPO_ROOT = _candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.device import log_device_info
device = log_device_info()


## Method and Outputs

### Method

1. Load `EvalRecord` rows from the baseline eval run (`#19`) via `read_eval_records_jsonl`.
2. Call `classify_error()` and `mark_recoverability()` on every record to build `TrajectoryRow` objects.
3. Produce category × error-type cross-tabulation.
4. Call `produce_retry_candidates()` — incorrect rows that have a boxed-format answer.
5. Write `data/analysis/retry_candidates.jsonl` and `data/analysis/trajectories_<run_id>.jsonl`.
6. Write `docs/analysis/error_slices.md` — per-category accuracy + top error modes.

### Expected outputs

- `data/analysis/trajectories_<run_id>.jsonl` — one row per EvalRecord with error_type and recoverability fields
- `data/analysis/retry_candidates.jsonl` — subset of recoverable failures for synthetic data seeding
- `docs/analysis/error_slices.md` — taxonomy table with category-level accuracy and error-mode counts

In [ ]:
import json
from pathlib import Path

from src.evaluation.records import read_eval_records_jsonl
from src.evaluation.trajectory import (
    ErrorType,
    build_trajectory_rows,
    produce_retry_candidates,
    slice_by_category,
    slice_by_error_type,
    write_retry_candidates,
    write_trajectory_jsonl,
)

# ── Set this to the actual baseline eval_records path from issue #19 ──────────
# Example: data/eval/runs/baseline-v1/eval_records.jsonl
BASELINE_PATH = Path("data/eval/runs/baseline-v1/eval_records.jsonl")

print(f"Loading EvalRecords from: {BASELINE_PATH}")
if not BASELINE_PATH.exists():
    print("WARNING: baseline path not found — using empty list for scaffold demo.")
    records = []
else:
    records = read_eval_records_jsonl(BASELINE_PATH)

print(f"Loaded {len(records)} EvalRecord(s)")

In [ ]:
# Build TrajectoryRows from EvalRecords
rows = build_trajectory_rows(records)
candidates = produce_retry_candidates(rows)

print(f"Total rows:       {len(rows)}")
print(f"Retry candidates: {len(candidates)}")

# Error-type distribution
by_error = slice_by_error_type(rows)
print("\nError type distribution:")
for et, rlist in sorted(by_error.items(), key=lambda kv: -len(kv[1])):
    print(f"  {et:<30} {len(rlist)}")

In [ ]:
# Category × error-type cross-tabulation
by_category = slice_by_category(rows)

print("Category × error-type cross-tab:")
all_error_types = [et.value for et in ErrorType]
header = f"{'Category':<20}" + "".join(f"  {et:<12}" for et in all_error_types)
print(header)
print("-" * len(header))
for cat in sorted(by_category):
    by_et = slice_by_error_type(by_category[cat])
    row_str = f"{cat:<20}" + "".join(f"  {len(by_et.get(et, [])):<12}" for et in all_error_types)
    print(row_str)

In [ ]:
# Write artifacts
run_id = records[0].run_id if records else "no-run"

trajectories_path = Path("data/analysis") / f"trajectories_{run_id}.jsonl"
candidates_path = Path("data/analysis/retry_candidates.jsonl")

write_trajectory_jsonl(rows, trajectories_path)
write_retry_candidates(candidates, candidates_path)

print(f"Written: {trajectories_path}  ({len(rows)} rows)")
print(f"Written: {candidates_path}  ({len(candidates)} retry candidates)")

In [ ]:
# Write docs/analysis/error_slices.md
import datetime

error_slices_path = Path("docs/analysis/error_slices.md")
error_slices_path.parent.mkdir(parents=True, exist_ok=True)

total = len(rows)
correct_count = sum(1 for r in rows if r.record.correct)
accuracy = correct_count / total if total else 0.0

lines = [
    "# Error Slice Taxonomy",
    "",
    f"Generated: {datetime.date.today()}  ",
    f"Source run: `{run_id}`  ",
    f"Total examples: {total}  ",
    f"Overall accuracy: {accuracy:.1%}",
    "",
    "## Error-Type Distribution",
    "",
    "| Error Type | Count | % of Total |",
    "|---|---:|---:|",
]
for et, rlist in sorted(by_error.items(), key=lambda kv: -len(kv[1])):
    pct = len(rlist) / total * 100 if total else 0
    lines.append(f"| `{et}` | {len(rlist)} | {pct:.1f}% |")

lines += [
    "",
    "## Per-Category Accuracy",
    "",
    "| Category | Total | Correct | Accuracy |",
    "|---|---:|---:|---:|",
]
for cat in sorted(by_category):
    cat_rows = by_category[cat]
    cat_correct = sum(1 for r in cat_rows if r.record.correct)
    cat_acc = cat_correct / len(cat_rows) if cat_rows else 0.0
    lines.append(f"| {cat} | {len(cat_rows)} | {cat_correct} | {cat_acc:.1%} |")

lines += [
    "",
    "## Retry Candidates",
    "",
    f"Recoverable failures (boxed format present, wrong answer): **{len(candidates)}**",
    "",
    f"Artifact: `data/analysis/retry_candidates.jsonl`",
]

error_slices_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Written: {error_slices_path}")

In [ ]:
# Verification assertions
correct_rows = [r for r in rows if r.record.correct]
incorrect_rows = [r for r in rows if not r.record.correct]
assert len(correct_rows) + len(incorrect_rows) == len(rows), "Row count mismatch"

top_3 = sorted(by_error.items(), key=lambda kv: -len(kv[1]))[:3]
if rows:
    for et_name, et_rows in top_3:
        assert len(et_rows) >= 5, f"Top error type {et_name!r} has fewer than 5 examples"

print(f"Verification passed: {len(rows)} total rows, {len(candidates)} retry candidates")
print(f"Top 3 error types: {[et for et, _ in top_3]}")

## Results / Open Risks

**Artifacts produced:**
- `data/analysis/trajectories_<run_id>.jsonl` — full trajectory dataset
- `data/analysis/retry_candidates.jsonl` — recoverable failures for synthetic seeding
- `docs/analysis/error_slices.md` — category × error-type taxonomy report

**Open risks:**
- Requires real baseline `eval_records.jsonl` from `#19`; stub demo runs with empty list.
- Error classification is heuristic — borderline cases (short wrong numeric answers) may swap between `arithmetic_slip` and `hallucinated_reasoning`.
- `retry_candidates.jsonl` is consumed by Branch 3 (`#24`); check path before running synthetic generation.

## Sources

            - [Adversarial review](docs/analysis/ADVERSARIAL_REVIEW.md)
- [Trajectory dataset reference](https://www.kaggle.com/datasets/kishanvavdara/nemotron-reasoning-traj)
- [Prompting notebook](notebooks/05_prompting_and_decode_sweeps.ipynb)
